# Descriptive Statistics: Clinical Data

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')  # go up to scripts/ root



import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib.pyplot as plt
import scipy.stats as stats


df_anthropometrics = pd.read_csv('../../output/1_feature_extraction/df_anthropometric_data_2026-07-09.csv')
df_matsuda_index = pd.read_csv("../../data/insulin_resistance/IR_metrics.csv")[['study_id', 'matsuda_2h']]

df_anthropometrics = df_anthropometrics.drop(columns=['visit_name'])
display(df_anthropometrics.head())
print(df_anthropometrics[['study_id']].nunique())

display(df_matsuda_index.head())


# Functions

In [ ]:
def check_normality(data, feature_name="Feature", ax=None):
    """
    Generate Q-Q plot and run Shapiro-Wilk test for a given array/series.
    
    Parameters
    ----------
    data : array-like
        Numeric data to test (NaNs are dropped automatically).
    feature_name : str
        Label used in the plot title and printed output.
    ax : matplotlib axis, optional
        If provided, plots on this axis (useful for subplots/loops).
        Otherwise creates its own figure.
    
    Returns
    -------
    dict with W statistic, p-value, and a boolean for normality at alpha=0.05
    """
    data = pd.Series(data).dropna().values

    # Shapiro-Wilk test
    W, p_value = stats.shapiro(data)
    is_normal = p_value > 0.05

    # Q-Q plot
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: {feature_name}\nShapiro-Wilk W={W:.3f}, p={p_value:.4f}")

    print(f"{feature_name}: W={W:.4f}, p={p_value:.4f} -> "
          f"{'Normal (fail to reject H0)' if is_normal else 'Non-normal (reject H0)'}")

    return {"feature": feature_name, "W": W, "p_value": p_value, "is_normal": is_normal}



# --- Example: loop over multiple features in a feature matrix ---
def check_normality_batch(df, feature_cols, ncols=4):
    n = len(feature_cols)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    results = []
    for i, col in enumerate(feature_cols):
        data = df[col].dropna()
        W, p_value = stats.shapiro(data)
        skewness = stats.skew(data, bias=False)

        if abs(skewness) < 0.5:
            direction = "symmetric"
        elif skewness >= 0.5:
            direction = "right-skewed"
        else:
            direction = "left-skewed"

        stats.probplot(data, dist="norm", plot=axes[i])
        axes[i].set_title(f"{col}\nW={W:.3f}, p={p_value:.4f}\nskew={skewness:.2f} ({direction})")

        results.append({
            "feature": col, "W": W, "p_value": p_value,
            "is_normal": p_value > 0.05,
            "skewness": skewness, "direction": direction
        })

    for j in range(len(feature_cols), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    return pd.DataFrame(results)

## Calculate Descriptive Statistics

In [ ]:

df = df_anthropometrics.merge(df_matsuda_index, on='study_id', how='left')

#calculate statistics for each visit, with 0.75 and 0.25 quantiles
def first_q(x):
    return x.quantile(0.25)
first_q.__name__ = 'first_q'

def third_q(x):
    return x.quantile(0.75)
third_q.__name__ = 'third_q'

stats_clinical = df.agg({
    'height_m': ['mean', 'std', 'median', 'min',  'max', first_q, third_q],
    'waist_circ_cm': ['mean', 'std', 'median', 'min', 'max', first_q, third_q],
    'weight_kg': ['mean', 'std', 'median', 'min',  'max', first_q, third_q],
    'bmi': ['mean', 'std', 'median', 'min', 'max', first_q, third_q],
    "WHtR": ['mean', 'std', 'median', 'min', 'max', first_q, third_q],
    'age_at_visit': ['mean', 'std', 'median', 'min', 'max', first_q, third_q],
    'matsuda_2h': ['mean', 'std', 'median', 'min', 'max', first_q, third_q]
}).reset_index()

display(stats_clinical)

#save stats for V0 to csv
date = datetime.now().strftime("%Y-%m-%d")
stats_clinical.to_csv(f"../../output/2_descriptive_stats/clinical_data_{date}.csv", index=False)


### Q-Q Plot & Shapiro-Wilk Test

In [ ]:
clinical_cols = [
    'height_m',
    'weight_kg',
    'waist_circ_cm',
    'WHtR',
    'bmi',
    'age_at_visit', 
    'matsuda_2h']
clinical_summary = check_normality_batch(df, clinical_cols, ncols=5)
print(clinical_summary, f"\n")
date = datetime.now().strftime("%Y-%m-%d")
clinical_summary.to_csv(f"../../output/2_descriptive_stats/clinical_normality_summary_{date}.csv", index=False)